<a href="https://colab.research.google.com/github/Ololade117/Inducing-Polysemanticity/blob/main/The__flexible_bias.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

The Bias in simple terms offsets the curve in a way the Weight matrix cant. The goal of this research is to derive an approach towards having a more flexible bias(not just one value) that will be determined by the input, that can nudge the learning or prediction into the right direction easily with a lesser number of parameters(< weight matrix dim). The idea is to make a bias that does not contain just one number but several which could offset the learning better

The task:
1. Derive a means to make the weight matrix more flexible. I.e before y= wX +b , b  is obtained from B that contains different bs, b is most suited for X
2. Compare this with a model of similar paramter size

Hypothesis:
A model with a flexible bias(list of several biases)  will out perform a model with a fixed bias(one number)

# The Bias

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import random

In [4]:
with open('names.txt', 'r') as f:
    names = f.read().splitlines()

# Add a special start-of-sequence token and end-of-sequence token
# Using '.' for EOS and '^' for SOS for simplicity, assuming they are not in names
names = [f'^{name}.' for name in names]

# Build character vocabulary
chars = sorted(list(set(''.join(names))))
vocab_size = len(chars)

char_to_ix = {ch: i for i, ch in enumerate(chars)}
ix_to_char = {i: ch for i, ch in enumerate(chars)}

print(f'Vocabulary size: {vocab_size}')
print(f'Example names: {names[:5]}')

Vocabulary size: 28
Example names: ['^emma.', '^olivia.', '^ava.', '^isabella.', '^sophia.']


In [5]:
# Create training data (input, target pairs)
X = []
y = []

for name in names:
    for i in range(len(name) - 1):
        input_char = name[i]
        target_char = name[i+1]
        X.append(char_to_ix[input_char])
        y.append(char_to_ix[target_char])

X = torch.tensor(X, dtype=torch.long)
y = torch.tensor(y, dtype=torch.long)

print(f'Total training examples: {len(X)}')
print(f'First 5 input indices: {X[:5]}')
print(f'First 5 target indices: {y[:5]}')

Total training examples: 228146
First 5 input indices: tensor([ 1,  6, 14, 14,  2])
First 5 target indices: tensor([ 6, 14, 14,  2,  0])


In [13]:
# @title
import torch.nn as nn # Added import for robustness
import torch # Ensure torch is imported for tensor operations

class ManualFeedforwardNet(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim): # hidden_dim is kept in signature but not used for architecture
        super(ManualFeedforwardNet, self).__init__()
        self.vocab_size = vocab_size
        self.embedding_dim = embedding_dim

        # Manually create embedding table (C)
        self.C = torch.randn((vocab_size, embedding_dim), requires_grad=True)

        # A single linear layer mapping from embedding_dim directly to vocab_size
        # This replaces the two linear layers (W1, b1, W2, b2) with one output layer (W_out, b_out)
        self.W_out = torch.randn((embedding_dim, vocab_size), requires_grad=True)
        self.b_out = torch.randn(vocab_size, requires_grad=True)

        # Store parameters in a list for the optimizer
        self.parameters_list = [self.C, self.W_out, self.b_out]

    def forward(self, x):
        # x is a tensor of character indices
        # Embedding lookup
        emb = self.C[x] # (batch_size, embedding_dim)

        # Single linear layer (output layer) directly from embeddings
        logits = emb @ self.W_out + self.b_out # (batch_size, vocab_size)
        return logits

# Model parameters (using existing vocab_size, embedding_dim, hidden_dim)
embedding_dim = 10 # Defined here for robustness
hidden_dim = 128   # Defined here for robustness (though not used in this single-layer architecture)

# Instantiate the new model (will reuse the global vocab_size, embedding_dim, hidden_dim)
manual_model = ManualFeedforwardNet(vocab_size, embedding_dim, hidden_dim)

# Define loss function and optimizer for the manual model
criterion_manual = nn.CrossEntropyLoss()
optimizer_manual = optim.Adam(manual_model.parameters_list, lr=0.01)

print(manual_model)
print(f'Total number of manually created parameters: {sum(p.numel() for p in manual_model.parameters_list)}')

# Assign the manual model to 'model' so subsequent training cells can use it
normal_model = manual_model
criterion = criterion_manual
optimizer = optimizer_manual

ManualFeedforwardNet()
Total number of manually created parameters: 588


In [16]:
num_epochs = 20
batch_size = 64

# Create a DataLoader for batching
dataset = torch.utils.data.TensorDataset(X, y)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

print("Starting training of Normal Model...")
for epoch in range(num_epochs):
    total_loss = 0
    for inputs, targets in dataloader:
        optimizer.zero_grad() # Zero the gradients
        outputs = normal_model(inputs) # Forward pass
        loss = criterion(outputs, targets) # Calculate loss
        loss.backward() # Backward pass
        optimizer.step() # Update weights
        total_loss += loss.item()

    print(f'Epoch {epoch+1}/{num_epochs}, Loss: {total_loss/len(dataloader):.4f}')

print("Training finished!")

Starting training of Normal Model...
Epoch 1/20, Loss: 2.4773
Epoch 2/20, Loss: 2.4774
Epoch 3/20, Loss: 2.4772
Epoch 4/20, Loss: 2.4777
Epoch 5/20, Loss: 2.4773
Epoch 6/20, Loss: 2.4777
Epoch 7/20, Loss: 2.4775
Epoch 8/20, Loss: 2.4773
Epoch 9/20, Loss: 2.4775
Epoch 10/20, Loss: 2.4777
Epoch 11/20, Loss: 2.4773
Epoch 12/20, Loss: 2.4773
Epoch 13/20, Loss: 2.4776
Epoch 14/20, Loss: 2.4775
Epoch 15/20, Loss: 2.4775
Epoch 16/20, Loss: 2.4778
Epoch 17/20, Loss: 2.4771
Epoch 18/20, Loss: 2.4772
Epoch 19/20, Loss: 2.4771
Epoch 20/20, Loss: 2.4774
Training finished!


In [11]:
import torch.nn as nn # Added import for robustness
import torch # Ensure torch is imported for tensor operations

class FlexibleBiasNet(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim): # hidden_dim is kept in signature but not used for architecture
        super(FlexibleBiasNet, self).__init__()
        self.vocab_size = vocab_size
        self.embedding_dim = embedding_dim

        # Manually create embedding table (C)
        self.C = torch.randn((vocab_size, embedding_dim), requires_grad=True)

        # A single linear layer mapping from embedding_dim directly to vocab_size
        # This replaces the two linear layers (W1, b1, W2, b2) with one output layer (W_out, b_out)
        self.W_out = torch.randn((embedding_dim, vocab_size), requires_grad=True)
        self.B_out = torch.randn((embedding_dim, 1), requires_grad=True)
        #self.b_out = torch.randn(vocab_size, requires_grad=True)

        # Store parameters in a list for the optimizer
        self.parameters_list = [self.C, self.W_out, self.B_out]

    def forward(self, x):
        # x is a tensor of character indices
        # Embedding lookup
        emb = self.C[x] # (batch_size, embedding_dim)
        b = emb @ self.B_out

        # Single linear layer (output layer) directly from embeddings
        logits = emb @ self.W_out + b # (batch_size, vocab_size)
        return logits

# Model parameters (using existing vocab_size, embedding_dim, hidden_dim)
embedding_dim = 10 # Defined here for robustness
hidden_dim = 128   # Defined here for robustness (though not used in this single-layer architecture)

# Instantiate the new model (will reuse the global vocab_size, embedding_dim, hidden_dim)
manual_model = FlexibleBiasNet(vocab_size, embedding_dim, hidden_dim)

# Define loss function and optimizer for the manual model
criterion_manual = nn.CrossEntropyLoss()
optimizer_manual = optim.Adam(manual_model.parameters_list, lr=0.01)

print(manual_model)
print(f'Total number of manually created parameters: {sum(p.numel() for p in manual_model.parameters_list)}')

# Assign the manual model to 'model' so subsequent training cells can use it
model = manual_model
criterion = criterion_manual
optimizer = optimizer_manual

FlexibleBiasNet()
Total number of manually created parameters: 570


In [15]:
num_epochs = 20
batch_size = 64

# Create a DataLoader for batching
dataset = torch.utils.data.TensorDataset(X, y)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

print("Starting training of FlexibleBiasNet...")
for epoch in range(num_epochs):
    total_loss = 0
    for inputs, targets in dataloader:
        optimizer.zero_grad() # Zero the gradients
        outputs = model(inputs) # Forward pass
        loss = criterion(outputs, targets) # Calculate loss
        loss.backward() # Backward pass
        optimizer.step() # Update weights
        total_loss += loss.item()

    print(f'Epoch {epoch+1}/{num_epochs}, Loss: {total_loss/len(dataloader):.4f}')

print("Training finished!")

Starting training of FlexibleBiasNet...
Epoch 1/20, Loss: 2.4782
Epoch 2/20, Loss: 2.4782
Epoch 3/20, Loss: 2.4782
Epoch 4/20, Loss: 2.4782
Epoch 5/20, Loss: 2.4782
Epoch 6/20, Loss: 2.4782
Epoch 7/20, Loss: 2.4782
Epoch 8/20, Loss: 2.4782
Epoch 9/20, Loss: 2.4782
Epoch 10/20, Loss: 2.4782
Epoch 11/20, Loss: 2.4782
Epoch 12/20, Loss: 2.4782
Epoch 13/20, Loss: 2.4782
Epoch 14/20, Loss: 2.4782
Epoch 15/20, Loss: 2.4782
Epoch 16/20, Loss: 2.4782
Epoch 17/20, Loss: 2.4782
Epoch 18/20, Loss: 2.4782
Epoch 19/20, Loss: 2.4782
Epoch 20/20, Loss: 2.4782
Training finished!


# The normal FFN

### Load and Preprocess Data

First, we'll load the `names.txt` file, build a character vocabulary, and create training examples. Each example will consist of an input character and its corresponding next character.

Next, we'll prepare the dataset for training. For each name, we create sequences of (input character, target character) pairs. The input will be one-hot encoded.

### Define the Feedforward Network Model

Now we'll define the neural network architecture. It will be a simple feedforward network with an embedding layer to convert character indices into dense vectors, followed by a linear layer, an activation function (ReLU), and a final linear layer to output logits for each character in the vocabulary.

In [4]:
class FeedforwardNet(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super(FeedforwardNet, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.fc1 = nn.Linear(embedding_dim, hidden_dim) # First linear layer
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, vocab_size)    # Output linear layer

    def forward(self, x):
        x = self.embedding(x)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

# Model parameters
embedding_dim = 10
hidden_dim = 128

# Instantiate the model
model = FeedforwardNet(vocab_size, embedding_dim, hidden_dim)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

print(model)
print(f'Total number of parameters: {sum(p.numel() for p in model.parameters())}')

FeedforwardNet(
  (embedding): Embedding(28, 10)
  (fc1): Linear(in_features=10, out_features=128, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=128, out_features=28, bias=True)
)
Total number of parameters: 5300


A one Layer neural net with flexible neuron

In [7]:
# @title
class ManualFeedforwardNet(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super(ManualFeedforwardNet, self).__init__()
        self.vocab_size = vocab_size
        self.embedding_dim = embedding_dim
        self.hidden_dim = hidden_dim

        # Manually create embedding table (C)
        self.C = torch.randn((vocab_size, embedding_dim), requires_grad=True)

        # Manually create weights and biases for the first linear layer (W1, b1)
        self.W1 = torch.randn((embedding_dim, hidden_dim), requires_grad=True)
        self.b1 = torch.randn(hidden_dim, requires_grad=True)

        # Manually create weights and biases for the second linear layer (W2, b2)
        self.W2 = torch.randn((hidden_dim, vocab_size), requires_grad=True)
        self.b2 = torch.randn(vocab_size, requires_grad=True)

        # Store parameters in a list for the optimizer
        self.parameters_list = [self.C, self.W1, self.b1, self.W2, self.b2]

    def forward(self, x):
        # x is a tensor of character indices
        # Embedding lookup
        emb = self.C[x] # (batch_size, embedding_dim)

        # First linear layer + ReLU activation
        h = torch.tanh(emb @ self.W1 + self.b1) # (batch_size, hidden_dim)

        # Second linear layer (output layer)
        logits = h @ self.W2 + self.b2 # (batch_size, vocab_size)
        return logits

# Model parameters (using existing vocab_size, embedding_dim, hidden_dim)

# Instantiate the new model
manual_model = ManualFeedforwardNet(vocab_size, embedding_dim, hidden_dim)

# Define loss function and optimizer for the manual model
criterion_manual = nn.CrossEntropyLoss()
optimizer_manual = optim.Adam(manual_model.parameters_list, lr=0.01)

print(manual_model)
print(f'Total number of manually created parameters: {sum(p.numel() for p in manual_model.parameters_list)}')

# Assign the manual model to 'model' so subsequent training cells can use it
model = manual_model
criterion = criterion_manual
optimizer = optimizer_manual

ManualFeedforwardNet()
Total number of manually created parameters: 5300


In [9]:
# @title
manual_model.W1.shape, manual_model.b1.shape, manual_model.C.shape

(torch.Size([10, 128]), torch.Size([128]), torch.Size([28, 10]))

### Train the Model

We will now train the `FeedforwardNet` on our dataset of character pairs. The model will learn to predict the next character given the current one.

In [5]:
num_epochs = 20
batch_size = 64

# Create a DataLoader for batching
dataset = torch.utils.data.TensorDataset(X, y)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

print("Starting training...")
for epoch in range(num_epochs):
    total_loss = 0
    for inputs, targets in dataloader:
        optimizer.zero_grad() # Zero the gradients
        outputs = model(inputs) # Forward pass
        loss = criterion(outputs, targets) # Calculate loss
        loss.backward() # Backward pass
        optimizer.step() # Update weights
        total_loss += loss.item()

    print(f'Epoch {epoch+1}/{num_epochs}, Loss: {total_loss/len(dataloader):.4f}')

print("Training finished!")

Starting training...
Epoch 1/20, Loss: 2.4931
Epoch 2/20, Loss: 2.4748
Epoch 3/20, Loss: 2.4722
Epoch 4/20, Loss: 2.4700
Epoch 5/20, Loss: 2.4698
Epoch 6/20, Loss: 2.4695
Epoch 7/20, Loss: 2.4688
Epoch 8/20, Loss: 2.4686
Epoch 9/20, Loss: 2.4688
Epoch 10/20, Loss: 2.4690
Epoch 11/20, Loss: 2.4690
Epoch 12/20, Loss: 2.4695
Epoch 13/20, Loss: 2.4695
Epoch 14/20, Loss: 2.4685
Epoch 15/20, Loss: 2.4697
Epoch 16/20, Loss: 2.4704
Epoch 17/20, Loss: 2.4698
Epoch 18/20, Loss: 2.4701
Epoch 19/20, Loss: 2.4717
Epoch 20/20, Loss: 2.4708
Training finished!


### Generate Names with the Trained Model

After training, we can use the model to generate new names by sampling characters sequentially based on the predicted probabilities.

In [6]:
def generate_name(model, start_char='^', max_len=20):
    model.eval() # Set model to evaluation mode
    name = [start_char]
    input_idx = char_to_ix[start_char]

    with torch.no_grad():
        for _ in range(max_len):
            input_tensor = torch.tensor([input_idx], dtype=torch.long)
            output = model(input_tensor) # Get logits
            probabilities = torch.softmax(output, dim=1) # Convert to probabilities
            # Sample the next character based on probabilities
            next_char_idx = torch.multinomial(probabilities, num_samples=1).item()

            next_char = ix_to_char[next_char_idx]
            if next_char == '.': # End of sequence token
                break
            name.append(next_char)
            input_idx = next_char_idx

    return ''.join(name[1:]) # Exclude the start_char

print("Generated names:")
for _ in range(10):
    print(generate_name(model))

Generated names:
chievab
lyn
zie
mahmerya
ahayzaholamolynia
kya
ph
gedad
lemanckhurarsprje
stumini
